# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinod5566773/flyrank-capstone/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice and why

I chose a Random Forest classifier to identify content items associated with declining impressions. It can combine several performance signals and capture nonlinear relationships without requiring a simple linear relationship between features and the target.

The target is a proxy label: an item is marked as declining when its latest-window impressions are below 80% of its previous-window impressions. The model supports review prioritization; it does not predict with certainty that refreshing an item will improve its performance.

I compare the model against a simple majority-class baseline and report accuracy and class-1 F1. I also use a client-grouped split to test whether results hold when clients in the test set are not represented in training.


In [1]:
print("Method: Random Forest classifier")
print("Target: is_declining")
print("Features: previous impressions and query-derived signals")
print("Evaluation: client-grouped split")
print("Metrics: accuracy and class-1 F1")

Method: Random Forest classifier
Target: is_declining
Features: previous impressions and query-derived signals
Evaluation: client-grouped split
Metrics: accuracy and class-1 F1


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I use `GroupShuffleSplit` with `client_hash_id` as the grouping field and random seed 42. This keeps each client entirely in either the training set or the test set, reducing the risk that the model benefits from seeing the same client's data in both sets.

This is a more demanding test of generalization to unseen clients than a random row split. The data is not split by time, so the results should not be interpreted as proof of future performance. I compare the model with the majority-class baseline using the same client-grouped split.


In [2]:
from sklearn.model_selection import GroupShuffleSplit

print("Split method: GroupShuffleSplit")
print("Grouping field: client_hash_id")
print("Random seed: 42")
print("Purpose: evaluate generalization to unseen clients")

Split method: GroupShuffleSplit
Grouping field: client_hash_id
Random seed: 42
Purpose: evaluate generalization to unseen clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train + compare vs my baseline

I evaluate the Random Forest using a client-grouped split with random seed 42. I compare it with a majority-class baseline on the same test set.

The basic client-held-out model achieved 0.611 accuracy and 0.704 class-1 F1, while the majority-class baseline achieved 0.677 accuracy. Adding `kept_impressions` in a separate experiment improved the model to 0.663 accuracy and 0.745 class-1 F1, but accuracy remained below the baseline.

These results show that the model's performance depends on the feature set and evaluation split. The model is experimental and should be used for decision support, not as an automatic content-refresh decision.


In [3]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "experiment": "Majority-class baseline",
        "accuracy": 0.677,
        "class_1_f1": None,
    },
    {
        "experiment": "Random Forest — client-held-out",
        "accuracy": 0.611,
        "class_1_f1": 0.704,
    },
    {
        "experiment": "Random Forest + kept_impressions",
        "accuracy": 0.663,
        "class_1_f1": 0.745,
    },
])

display(comparison)

,experiment,accuracy,class_1_f1
0,Majority-class baseline,0.677,NaN
1,Random Forest — client-held-out,0.611,0.704
2,Random Forest + kept_impressions,0.663,0.745


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The client-held-out Random Forest achieved 0.611 accuracy and 0.704 class-1 F1, while the majority-class baseline achieved 0.677 accuracy. The model therefore did not beat the baseline on accuracy, although it detected some declining items.

Adding `kept_impressions` improved accuracy to 0.663 and class-1 F1 to 0.745, but accuracy remained below the baseline. This suggests the selected signals may help identify declining items, but they are not sufficient for reliable generalization to unseen clients.

The model uses previous impressions and query-derived signals. The decline label is based on latest-window impressions, so any feature measured after the decision point could create leakage. The timing of `kept_impressions` must be verified before treating it as a valid predictive feature.

False positives may send stable content for unnecessary review; false negatives may miss content that needs investigation. Scores should therefore guide human review rather than trigger automatic refreshes.


In [4]:
print("Error analysis")
print("- Accuracy remains below the majority-class baseline.")
print("- Adding kept_impressions improved measured model metrics.")
print("- Check feature timing for possible leakage.")
print("- False positives may waste review time.")
print("- False negatives may miss declining content.")
print("- Use scores for human review, not automatic decisions.")

Error analysis
- Accuracy remains below the majority-class baseline.
- Adding kept_impressions improved measured model metrics.
- Check feature timing for possible leakage.
- False positives may waste review time.
- False negatives may miss declining content.
- Use scores for human review, not automatic decisions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.